In [1]:
# Install websocket client library (if needed)
import sys
!{sys.executable} -m pip install -q websocket-client

## Subscribe to `level2` for BTC-USD

The `level2` channel sends:
- A `snapshot` message with full bids/asks
- Subsequent `l2update` messages with `[side, price, size]` changes

A `size` of `"0"` means remove that price level.

In [2]:
import json
from websocket import WebSocketApp

WS_URL = "wss://ws-feed.exchange.coinbase.com"
PRODUCT_IDS = ["BTC-USD"]
CHANNELS = ["level2"]

max_updates = 15
state = {"count": 0, "got_snapshot": False}

def on_open(ws):
    subscribe_msg = {
        "type": "subscribe",
        "product_ids": PRODUCT_IDS,
        "channels": CHANNELS,
    }
    ws.send(json.dumps(subscribe_msg))
    print("subscribed:", subscribe_msg)

def on_message(ws, message):
    msg = json.loads(message)
    msg_type = msg.get("type")

    if msg_type == "snapshot":
        state["got_snapshot"] = True
        print("snapshot bids", len(msg.get("bids", [])), "asks", len(msg.get("asks", [])))
        return

    if msg_type == "l2update":
        state["count"] += 1
        print(msg)
        if state["count"] >= max_updates:
            ws.close()
        return

    if msg_type == "subscriptions":
        print("subscriptions confirmed")
        return

    # Ignore other message types
    return

def on_error(ws, error):
    print("error:", error)

def on_close(ws, status_code, msg):
    print("closed:", status_code, msg)

ws = WebSocketApp(
    WS_URL,
    on_open=on_open,
    on_message=on_message,
    on_error=on_error,
    on_close=on_close,
)

ws.run_forever(ping_interval=30, ping_timeout=10)

subscribed: {'type': 'subscribe', 'product_ids': ['BTC-USD'], 'channels': ['level2']}
subscriptions confirmed
error: 
closed: None None
error: 
closed: None None


True